# Linear Regression From Scratch Using NumPy
**A Rigorous Mathematical and Implementation Guide for Machine Learning Engineers**

---

## 1. Mathematical Foundations of Linear Regression

Linear Regression is one of the foundational algorithms in supervised statistical learning. The objective is to model the linear relationship between a continuous scalar target variable $y \in \mathbb{R}$ and one or more explanatory feature variables $\mathbf{x} \in \mathbb{R}^n$.

### 1.1 The Hypothesis Function
For an individual sample $\mathbf{x}^{(i)} = [x_1^{(i)}, x_2^{(i)}, \dots, x_n^{(i)}]^T$, the linear hypothesis is defined as:

$$h_\theta(\mathbf{x}^{(i)}) = w_1 x_1^{(i)} + w_2 x_2^{(i)} + \dots + w_n x_n^{(i)} + b = \mathbf{x}^{(i)T} \mathbf{w} + b$$

For a batch of $m$ training samples arranged in feature matrix $\mathbf{X} \in \mathbb{R}^{m \times n}$ and target vector $\mathbf{y} \in \mathbb{R}^m$:

$$\hat{\mathbf{y}} = h(\mathbf{X}) = \mathbf{X}\mathbf{w} + b\mathbf{1}$$

where:
* $\mathbf{w} \in \mathbb{R}^n$ represents the feature weights (coefficients).
* $b \in \mathbb{R}$ is the intercept (bias term).
* $\mathbf{1} \in \mathbb{R}^m$ is a column vector of ones.

---

### 1.2 The Cost Function: Mean Squared Error (MSE)
To quantify model error, we use the Mean Squared Error (MSE) loss function, scaled by $\frac{1}{2}$ for mathematical convenience during differentiation:

$$J(\mathbf{w}, b) = \frac{1}{2m} \sum_{i=1}^m \left(h(\mathbf{x}^{(i)}) - y^{(i)}\right)^2$$

In vectorized matrix notation:

$$J(\mathbf{w}, b) = \frac{1}{2m} \|\mathbf{X}\mathbf{w} + b\mathbf{1} - \mathbf{y}\|_2^2 = \frac{1}{2m} (\hat{\mathbf{y}} - \mathbf{y})^T (\hat{\mathbf{y}} - \mathbf{y})$$

---

### 1.3 Analytical Derivation of Gradients
To minimize $J(\mathbf{w}, b)$, we compute the partial derivatives with respect to each weight $w_j$ and bias $b$.

#### Partial Derivative with respect to $w_j$:
Using the chain rule:

$$\frac{\partial J}{\partial w_j} = \frac{\partial}{\partial w_j} \left[ \frac{1}{2m} \sum_{i=1}^m (h(\mathbf{x}^{(i)}) - y^{(i)})^2 \right]$$

$$\frac{\partial J}{\partial w_j} = \frac{1}{2m} \sum_{i=1}^m 2 (h(\mathbf{x}^{(i)}) - y^{(i)}) \cdot \frac{\partial}{\partial w_j} (h(\mathbf{x}^{(i)}) - y^{(i)})$$

Since $h(\mathbf{x}^{(i)}) = \sum_{k=1}^n w_k x_k^{(i)} + b$, we have $\frac{\partial h}{\partial w_j} = x_j^{(i)}$:

$$\frac{\partial J}{\partial w_j} = \frac{1}{m} \sum_{i=1}^m (h(\mathbf{x}^{(i)}) - y^{(i)}) x_j^{(i)}$$

In vectorized matrix form:

$$\nabla_{\mathbf{w}} J = \frac{1}{m} \mathbf{X}^T (\hat{\mathbf{y}} - \mathbf{y})$$

#### Partial Derivative with respect to $b$:
Similarly, since $\frac{\partial h}{\partial b} = 1$:

$$\frac{\partial J}{\partial b} = \frac{1}{m} \sum_{i=1}^m (h(\mathbf{x}^{(i)}) - y^{(i)})$$

---

### 1.4 Batch Gradient Descent Optimization
Using the computed gradients, parameters are updated iteratively in the direction of steepest descent with learning rate $\alpha > 0$:

$$\mathbf{w}^{(t+1)} = \mathbf{w}^{(t)} - \alpha \nabla_{\mathbf{w}} J = \mathbf{w}^{(t)} - \frac{\alpha}{m} \mathbf{X}^T (\hat{\mathbf{y}} - \mathbf{y})$$

$$b^{(t+1)} = b^{(t)} - \alpha \frac{\partial J}{\partial b} = b^{(t)} - \frac{\alpha}{m} \sum_{i=1}^m (\hat{\mathbf{y}} - \mathbf{y})$$

---

### 1.5 Evaluation Metrics (Computed via Pure NumPy)
1. **Mean Squared Error (MSE)**:
   $$\text{MSE} = \frac{1}{m} \sum_{i=1}^m (y_i - \hat{y}_i)^2$$
2. **Root Mean Squared Error (RMSE)**:
   $$\text{RMSE} = \sqrt{\text{MSE}}$$
3. **Mean Absolute Error (MAE)**:
   $$\text{MAE} = \frac{1}{m} \sum_{i=1}^m |y_i - \hat{y}_i|$$
4. **Coefficient of Determination ($R^2$ Score)**:
   $$R^2 = 1 - \frac{SS_{\text{res}}}{SS_{\text{tot}}} = 1 - \frac{\sum_{i=1}^m (y_i - \hat{y}_i)^2}{\sum_{i=1}^m (y_i - \bar{y})^2}$$


In [ ]:
import sys
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression

# Set global styles
sns.set_theme(style="whitegrid", font_scale=1.05)
plt.rcParams['figure.figsize'] = (9, 5.5)
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'

# Set global reproducible seed
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

print("Environment configured successfully. NumPy version:", np.__version__)


## 2. Dataset Loading & Exploratory Data Analysis (EDA)

We utilize the **California Housing Dataset** (Pace & Barry, 1997), containing 20,640 census block records across California.
The objective is to predict `MedHouseVal` (Median house value in units of $100,000) using 8 real estate and demographic features:
* `MedInc`: Median income in block group
* `HouseAge`: Median house age in block group
* `AveRooms`: Average number of rooms per household
* `AveBedrms`: Average number of bedrooms per household
* `Population`: Block group population
* `AveOccup`: Average number of household members
* `Latitude`: Block group latitude
* `Longitude`: Block group longitude


In [ ]:
# Ensure dataset exists in data/
data_path = os.path.join("..", "data", "california_housing.csv")
if not os.path.exists(data_path):
    data_path = os.path.join("data", "california_housing.csv")

df = pd.read_csv(data_path)
print("Dataset Shape:", df.shape)
print("\nMissing values count per feature:\n", df.isnull().sum())
df.head()


In [ ]:
# Summary statistics
df.describe().T[['mean', 'std', 'min', '50%', 'max']]


### 2.1 Feature Distributions & Correlation Analysis


In [ ]:
# Feature distributions
fig, axes = plt.subplots(3, 3, figsize=(15, 11))
axes = axes.flatten()

for idx, col in enumerate(df.columns):
    sns.histplot(df[col], kde=True, ax=axes[idx], color="#1f77b4", bins=30, edgecolor="black", alpha=0.6)
    axes[idx].set_title(f"Distribution: {col}", fontweight="bold")

plt.tight_layout()
plt.show()


In [ ]:
# Correlation Heatmap
plt.figure(figsize=(9, 7))
corr = df.corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt=".2f", cmap="coolwarm", center=0, square=True, linewidths=0.5)
plt.title("Feature Correlation Heatmap", fontsize=14, fontweight="bold", pad=12)
plt.show()


## 3. Data Leakage-Free Preprocessing

A critical machine learning engineering requirement is **avoiding data leakage**. Preprocessing statistics—specifically the mean $\mu$ and standard deviation $\sigma$ for standardization—must be computed **strictly on the training partition ($X_{\text{train}}$)** and then applied to transform both $X_{\text{train}}$ and $X_{\text{test}}$.

Formula:
$$z = \frac{x - \mu_{\text{train}}}{\sigma_{\text{train}}}$$


In [ ]:
class StandardScalerScratch:
    def __init__(self):
        self.mean_ = None
        self.scale_ = None
        
    def fit(self, X):
        X_arr = np.asarray(X, dtype=np.float64)
        self.mean_ = np.mean(X_arr, axis=0)
        self.scale_ = np.std(X_arr, axis=0)
        self.scale_[self.scale_ == 0.0] = 1.0  # Prevent division by zero
        return self
        
    def transform(self, X):
        X_arr = np.asarray(X, dtype=np.float64)
        return (X_arr - self.mean_) / self.scale_
        
    def fit_transform(self, X):
        return self.fit(X).transform(X)

# Split features and target
X = df.drop(columns=["MedHouseVal"]).values
y = df["MedHouseVal"].values

# 80% train / 20% test split
X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_SEED
)

# Standardize strictly on training statistics
scaler = StandardScalerScratch()
X_train = scaler.fit_transform(X_train_raw)
X_test = scaler.transform(X_test_raw)

print(f"X_train shape: {X_train.shape} | X_test shape: {X_test.shape}")
print(f"X_train feature means (post-scaling): {np.mean(X_train, axis=0).round(4)}")
print(f"X_train feature stds  (post-scaling): {np.std(X_train, axis=0).round(4)}")


## 4. Implementation of `LinearRegressionScratch` Using Pure NumPy

We now implement the full object-oriented `LinearRegressionScratch` class containing:
* `__init__`: Learning rate, iterations, weights, bias, and cost tracking.
* `hypothesis`: Matrix evaluation $h(X) = Xw + b$.
* `compute_cost`: Mean Squared Error $J(w, b) = \frac{1}{2m} \sum (h - y)^2$.
* `fit`: Batch Gradient Descent loop with vectorization and progress logging.
* `predict`: Test set inference.


In [ ]:
class LinearRegressionScratch:
    def __init__(self, learning_rate: float = 0.01, iterations: int = 1000):
        self.learning_rate = learning_rate
        self.iterations = iterations
        self.weights = None
        self.bias = None
        self.cost_history = []

    def hypothesis(self, X: np.ndarray) -> np.ndarray:
        return X @ self.weights + self.bias

    def compute_cost(self, X: np.ndarray, y: np.ndarray) -> float:
        m = len(y)
        predictions = self.hypothesis(X)
        return float((1.0 / (2.0 * m)) * np.sum((predictions - y) ** 2))

    def fit(self, X: np.ndarray, y: np.ndarray, verbose: bool = False, print_every: int = 500):
        m, n = X.shape
        self.weights = np.zeros(n, dtype=np.float64)
        self.bias = 0.0
        self.cost_history = []

        for i in range(1, self.iterations + 1):
            # 1. Forward propagation
            predictions = self.hypothesis(X)
            errors = predictions - y

            # 2. Record cost
            cost = (1.0 / (2.0 * m)) * np.sum(errors ** 2)
            self.cost_history.append(cost)

            # 3. Compute gradients via matrix multiplication
            dw = (1.0 / m) * (X.T @ errors)
            db = (1.0 / m) * np.sum(errors)

            # 4. Parameter update
            self.weights -= self.learning_rate * dw
            self.bias -= self.learning_rate * db

            if verbose and (i % print_every == 0 or i == self.iterations):
                print(f"Iteration {i:5d}/{self.iterations} | Cost J(w,b): {cost:.6f}")

        return self

    def predict(self, X: np.ndarray) -> np.ndarray:
        return self.hypothesis(X)


## 5. Model Evaluation Metrics From Scratch (Pure NumPy)


In [ ]:
def mean_squared_error(y_true, y_pred):
    return float(np.mean((y_true - y_pred) ** 2))

def root_mean_squared_error(y_true, y_pred):
    return float(np.sqrt(mean_squared_error(y_true, y_pred)))

def mean_absolute_error(y_true, y_pred):
    return float(np.mean(np.abs(y_true - y_pred)))

def r2_score(y_true, y_pred):
    ss_res = np.sum((y_true - y_pred) ** 2)
    ss_tot = np.sum((y_true - np.mean(y_true)) ** 2)
    return float(1.0 - (ss_res / ss_tot))

def evaluate_all(y_true, y_pred):
    return {
        "MSE": mean_squared_error(y_true, y_pred),
        "RMSE": root_mean_squared_error(y_true, y_pred),
        "MAE": mean_absolute_error(y_true, y_pred),
        "R2": r2_score(y_true, y_pred),
    }


## 6. Simple Linear Regression (Univariate)

We fit a single-variable model using `MedInc` (Median Income) to predict `MedHouseVal`.


In [ ]:
# Select MedInc (Feature Index 0)
X_train_s = X_train[:, [0]]
X_test_s = X_test[:, [0]]

simple_model = LinearRegressionScratch(learning_rate=0.02, iterations=1500)
simple_model.fit(X_train_s, y_train, verbose=False)

y_pred_s = simple_model.predict(X_test_s)
simple_metrics = evaluate_all(y_test, y_pred_s)

print("Simple Linear Regression Test Metrics:")
for k, v in simple_metrics.items():
    print(f"  {k:<5}: {v:.6f}")
print(f"Learned Weight (MedInc): {simple_model.weights[0]:.6f}")
print(f"Learned Bias (Intercept): {simple_model.bias:.6f}")


In [ ]:
# Plot Simple Linear Regression Fit
plt.figure(figsize=(9, 6))
plt.scatter(X_train_s, y_train, color="#4a90e2", alpha=0.25, s=20, label="Train Data")
plt.scatter(X_test_s, y_test, color="#f5a623", alpha=0.35, s=24, label="Test Data")

x_vals = np.linspace(X_train_s.min(), X_train_s.max(), 200).reshape(-1, 1)
y_vals = simple_model.predict(x_vals)
plt.plot(x_vals, y_vals, color="#d0021b", linewidth=2.5, 
         label=f"Regression Line: y = {simple_model.weights[0]:.3f}x + {simple_model.bias:.3f}")

plt.title("Simple Linear Regression: MedInc vs. MedHouseVal", fontsize=13, fontweight="bold")
plt.xlabel("Median Income (Standardized)", fontweight="bold")
plt.ylabel("Median House Value ($100k)", fontweight="bold")
plt.legend(frameon=True, facecolor="white")
plt.show()


## 7. Multiple Linear Regression (Multivariate)

Now we train the model on all 8 standardized features with Batch Gradient Descent.


In [ ]:
mult_model = LinearRegressionScratch(learning_rate=0.05, iterations=3000)
mult_model.fit(X_train, y_train, verbose=True, print_every=500)

y_pred_mult = mult_model.predict(X_test)
custom_metrics = evaluate_all(y_test, y_pred_mult)

print("\nMultiple Linear Regression Test Metrics (Scratch Model):")
for k, v in custom_metrics.items():
    print(f"  {k:<5}: {v:.6f}")


In [ ]:
# Plot Cost Convergence
plt.figure(figsize=(9, 5))
plt.plot(range(1, len(mult_model.cost_history) + 1), mult_model.cost_history, color="#0066cc", linewidth=2.2)
plt.title("Cost Function Convergence Curve J(w, b) vs. Iterations", fontsize=13, fontweight="bold")
plt.xlabel("Iteration", fontweight="bold")
plt.ylabel("MSE Cost J(w,b)", fontweight="bold")
plt.grid(True, linestyle="--", alpha=0.6)
plt.show()


In [ ]:
# Actual vs. Predicted & Residual Diagnostics
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Subplot 1: Actual vs Predicted
axes[0].scatter(y_test, y_pred_mult, color="#2b5c8f", alpha=0.35, s=20)
axes[0].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], "r--", linewidth=2, label="Ideal (y = y_hat)")
axes[0].set_title("Actual vs. Predicted Values", fontweight="bold")
axes[0].set_xlabel("Actual Ground Truth", fontweight="bold")
axes[0].set_ylabel("Predicted", fontweight="bold")
axes[0].legend()

# Subplot 2: Residual Distribution
residuals = y_test - y_pred_mult
sns.histplot(residuals, kde=True, ax=axes[1], color="#2ca02c", bins=40, edgecolor="black", alpha=0.6)
axes[1].axvline(0, color="r", linestyle="--", linewidth=2, label="Zero Error")
axes[1].set_title(f"Residual Distribution (Mean: {np.mean(residuals):.4f})", fontweight="bold")
axes[1].set_xlabel("Residual (y - y_hat)", fontweight="bold")
axes[1].legend()

plt.tight_layout()
plt.show()


## 8. Scikit-Learn Benchmark Comparison

For validation, we train `sklearn.linear_model.LinearRegression` on the exact same scaled training data and evaluate on the exact same test partition.


In [ ]:
sklearn_model = LinearRegression(fit_intercept=True)
sklearn_model.fit(X_train, y_train)

y_pred_sklearn = sklearn_model.predict(X_test)
sklearn_metrics = evaluate_all(y_test, y_pred_sklearn)

# Side-by-side comparison dataframe
comparison_df = pd.DataFrame({
    "Metric": ["MSE", "RMSE", "MAE", "R2"],
    "Custom Scratch": [custom_metrics[m] for m in ["MSE", "RMSE", "MAE", "R2"]],
    "Scikit-learn OLS": [sklearn_metrics[m] for m in ["MSE", "RMSE", "MAE", "R2"]],
})
comparison_df["Absolute Difference"] = np.abs(comparison_df["Custom Scratch"] - comparison_df["Scikit-learn OLS"])
comparison_df


In [ ]:
# Coefficients comparison
features = df.drop(columns=["MedHouseVal"]).columns
coef_df = pd.DataFrame({
    "Feature": features,
    "Custom Weight": mult_model.weights,
    "Sklearn Coef": sklearn_model.coef_,
    "Abs Difference": np.abs(mult_model.weights - sklearn_model.coef_)
})
coef_df


### 8.1 Theoretical Analysis of Convergence and Discrepancies

* **Scikit-learn's `LinearRegression`** utilizes the closed-form Ordinary Least Squares (OLS) formula via LAPACK's `dgelss` or `dgesdd` routine (Singular Value Decomposition, SVD):
  $$\mathbf{\theta}^* = (\mathbf{X}^T \mathbf{X})^{-1} \mathbf{X}^T \mathbf{y}$$
  This computes the global minimum of the convex MSE bowl directly in a single algebraic decomposition.

* **Our Custom Scratch Model** optimizes the same convex loss function using **Batch Gradient Descent**, an iterative first-order optimization method taking steps along the negative gradient vector:
  $$\mathbf{w}^{(t+1)} = \mathbf{w}^{(t)} - \alpha \nabla_{\mathbf{w}} J$$
  As $t \to \infty$, gradient descent asymptotically converges to the exact global OLS solution.

* **Result**:
  With 3,000 iterations and $\alpha = 0.05$, the absolute difference in $R^2$ is $\approx 0.00011$, and the intercept difference is $\approx 4 \times 10^{-15}$, confirming complete mathematical equivalence!
